# Master Kashmiri NLLB Translation Fine-Tuning (320,000+ Sentence Pairs)

This notebook fine-tunes **Meta NLLB-200** on a massive combined Kashmiri corpus including:
- **SMUQamar 270K** Kashmiri-English parallel dataset
- **Omarrran 49K** Kashmiri-English translation corpus
- **Kaggle Clean & Expanded Datasets**
- **JKBOSE Kashmiri School Textbooks & In-House Grammar Lessons**

### Instructions:
1. **Accelerator**: Select **GPU P100** or **GPU T4 x2**.
2. **Internet**: Toggle **Internet ON**.
3. **Datasets**: Attach `sameerahmadnajar/kashmiri-clean-parallel-dataset` and `sameerahmadnajar/kashmiri-expanded-parallel-dataset`.
4. Click **Run All**.


In [ ]:
# 1. Environment & GPU Verification
!nvidia-smi
!pip -q install 'transformers==4.48.3' 'peft==0.14.0' 'accelerate==1.3.0' 'datasets>=3.2' 'evaluate>=0.4' sacrebleu sentencepiece huggingface_hub


In [ ]:
# 2. Write Self-Contained Training Pipeline
from pathlib import Path
repo = Path("/kaggle/working/cipher")
repo.mkdir(parents=True, exist_ok=True)
script_dir = repo / "scripts"
script_dir.mkdir(parents=True, exist_ok=True)
(script_dir / "finetune-nllb-master.py").write_text("#!/usr/bin/env python3\n\"\"\"Master NLLB-200 Kashmiri-English Translation Fine-Tuning Pipeline.\n\nIngests and unifies:\n1. SMUQamar/Kashmiri-English-Dataset-270K (270k pairs)\n2. Omarrran/kashmiri_English_parallel_corpus_49K (49k pairs)\n3. Kaggle expanded & clean parallel datasets\n4. JKBOSE textbook stories and literary corpora\n5. 26 In-House verified Kashmiri grammar lessons (Wade/Koul structural alignment)\n\nModel: facebook/nllb-200-distilled-600M (or facebook/nllb-200-1.3B)\nLoRA: Rank 64, Alpha 128 on Attention Projections (q_proj, k_proj, v_proj, out_proj)\nEvaluation: sacreBLEU and chrF++ metrics with test smoke generations\n\"\"\"\n\nimport os\nimport sys\nimport csv\nimport glob\nimport json\nimport shutil\nimport hashlib\nimport unicodedata\nfrom pathlib import Path\n\nimport torch\nfrom datasets import Dataset, load_dataset\nfrom peft import LoraConfig, TaskType, get_peft_model\nfrom transformers import (\n    AutoModelForSeq2SeqLM,\n    AutoTokenizer,\n    DataCollatorForSeq2Seq,\n    Seq2SeqTrainer,\n    Seq2SeqTrainingArguments,\n)\nfrom transformers.trainer_utils import get_last_checkpoint\nimport evaluate\n\nROOT = Path(__file__).resolve().parents[1]\nMODEL_ID = os.getenv(\"BASE_MODEL\", \"facebook/nllb-200-distilled-600M\")\nOUTPUT_DIR = Path(os.getenv(\"OUTPUT_DIR\", \"/kaggle/working/kashmiri-nllb-master\"))\nHF_REPO_ID = os.getenv(\"HF_REPO_ID\", \"Sameer0313/kashmiri-nllb-lora\")\nHF_TOKEN = os.getenv(\"HF_TOKEN\", \"\")\n\nif not torch.cuda.is_available():\n    print(\"[!] Warning: CUDA GPU not detected. Training will be slow.\")\n    device = \"cpu\"\nelse:\n    device = \"cuda\"\n    gpu_name = torch.cuda.get_device_name(0)\n    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)\n    print(f\"[+] Active GPU: {gpu_name} ({vram:.1f} GB VRAM)\")\n\n# ---------------------------------------------------------------------------\n# Kashmiri Text Normalization & Filtering\n# ---------------------------------------------------------------------------\ndef clean_text(text: str) -> str:\n    if not isinstance(text, str):\n        return \"\"\n    text = unicodedata.normalize(\"NFKC\", text)\n    return \" \".join(text.strip().split())\n\ndef is_valid_pair(en: str, ks: str) -> bool:\n    if not en or not ks:\n        return False\n    en_words = en.split()\n    ks_words = ks.split()\n    if len(en_words) < 2 or len(en_words) > 180:\n        return False\n    if len(ks_words) < 2 or len(ks_words) > 180:\n        return False\n    # Ensure Kashmiri side contains Arabic/Perso-Arabic script characters\n    if not any(\"\\u0600\" <= c <= \"\\u06ff\" or \"\\u0750\" <= c <= \"\\u077f\" for c in ks):\n        return False\n    # Filter out identical source and target (untranslated copy-paste)\n    if en.strip().lower() == ks.strip().lower():\n        return False\n    return True\n\n# ---------------------------------------------------------------------------\n# Corpus Aggregation from All Sources\n# ---------------------------------------------------------------------------\ndef load_all_parallel_data():\n    all_rows = []\n    seen = set()\n\n    def add_row(src, tgt, tier, src_lang=\"eng_Latn\", tgt_lang=\"kas_Arab\"):\n        src_c = clean_text(src)\n        tgt_c = clean_text(tgt)\n        if is_valid_pair(src_c, tgt_c):\n            key = (src_lang, src_c.lower(), tgt_c)\n            if key not in seen:\n                seen.add(key)\n                all_rows.append({\n                    \"source\": src_c,\n                    \"target\": tgt_c,\n                    \"src_lang\": src_lang,\n                    \"tgt_lang\": tgt_lang,\n                    \"tier\": tier\n                })\n\n    print(\"[*] Ingesting local repository gold & verified parallel data...\")\n    # 1. Local gold parallel\n    for p in [\n        ROOT / \"training-data/approved-parallel.jsonl\",\n        ROOT / \"training-data/combined-parallel.jsonl\",\n        ROOT / \"training-data/reviewed-parallel.jsonl\",\n        ROOT / \"training-data/seerah-chapter1-parallel-review.jsonl\",\n    ]:\n        if p.exists():\n            for line in p.read_text(encoding=\"utf-8\").splitlines():\n                if not line.strip():\n                    continue\n                try:\n                    row = json.loads(line)\n                    en = row.get(\"english\") or row.get(\"source\") or row.get(\"en\")\n                    ks = row.get(\"kashmiri\") or row.get(\"target\") or row.get(\"ks\")\n                    if en and ks:\n                        add_row(en, ks, tier=\"repo-gold\")\n                except Exception:\n                    pass\n\n    # 2. In-house Grammar Lessons (Lessons 1 to 26)\n    for p in ROOT.glob(\"training-data/native-speaker-grammar-lesson-*.json\"):\n        try:\n            data = json.loads(p.read_text(encoding=\"utf-8\"))\n            pairs = data.get(\"pairs\") or data.get(\"sentences\") or data if isinstance(data, list) else []\n            for item in pairs:\n                if isinstance(item, dict):\n                    en = item.get(\"english\") or item.get(\"en\")\n                    ks = item.get(\"kashmiri\") or item.get(\"ks\")\n                    if en and ks:\n                        add_row(en, ks, tier=\"grammar-lesson\")\n        except Exception:\n            pass\n\n    # 3. Kaggle Input Datasets (if running on Kaggle)\n    kaggle_input = Path(\"/kaggle/input\")\n    if kaggle_input.exists():\n        print(\"[*] Ingesting Kaggle parallel datasets...\")\n        for csv_file in kaggle_input.glob(\"**/*.csv\"):\n            try:\n                with csv_file.open(encoding=\"utf-8\", errors=\"ignore\") as f:\n                    reader = csv.DictReader(f)\n                    for r in reader:\n                        en = r.get(\"English\") or r.get(\"english\") or r.get(\"en\") or r.get(\"src\")\n                        ks = r.get(\"Kashmiri\") or r.get(\"kashmiri\") or r.get(\"ks\") or r.get(\"tgt\")\n                        if en and ks:\n                            add_row(en, ks, tier=\"kaggle-csv\")\n            except Exception:\n                pass\n        for jsonl_file in kaggle_input.glob(\"**/*.jsonl\"):\n            try:\n                for line in jsonl_file.read_text(encoding=\"utf-8\", errors=\"ignore\").splitlines():\n                    if line.strip():\n                        r = json.loads(line)\n                        en = r.get(\"english\") or r.get(\"en\") or r.get(\"source\")\n                        ks = r.get(\"kashmiri\") or r.get(\"ks\") or r.get(\"target\")\n                        if en and ks:\n                            add_row(en, ks, tier=\"kaggle-jsonl\")\n            except Exception:\n                pass\n\n    # 4. Hugging Face Large Corpora\n    print(\"[*] Ingesting Hugging Face online corpora (SMUQamar 270K & Omarrran 49K)...\")\n    for hf_id, split in [\n        (\"Omarrran/kashmiri_English_parallel_corpus_49K\", \"train\"),\n        (\"SMUQamar/Kashmiri-English-Dataset-270K\", \"train\"),\n    ]:\n        try:\n            print(f\"[*] Loading {hf_id}...\")\n            ds = load_dataset(hf_id, split=split, trust_remote_code=True)\n            for item in ds:\n                en = item.get(\"English\") or item.get(\"english\") or item.get(\"en\") or item.get(\"source\")\n                ks = item.get(\"Kashmiri\") or item.get(\"kashmiri\") or item.get(\"ks\") or item.get(\"target\")\n                if en and ks:\n                    add_row(en, ks, tier=hf_id)\n            print(f\"[+] Successfully loaded from {hf_id}!\")\n        except Exception as e:\n            print(f\"[-] Could not load {hf_id}: {e}\")\n\n    print(f\"[+] Total Verified & Deduplicated Parallel Sentences: {len(all_rows)}\")\n    return all_rows\n\n# ---------------------------------------------------------------------------\n# Training Pipeline\n# ---------------------------------------------------------------------------\nprint(f\"[*] Initializing tokenizer and model: {MODEL_ID}...\")\ntokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\nbase_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)\n\npeft_config = LoraConfig(\n    task_type=TaskType.SEQ_2_SEQ_LM,\n    r=64,\n    lora_alpha=128,\n    lora_dropout=0.05,\n    target_modules=[\"q_proj\", \"v_proj\", \"k_proj\", \"out_proj\"],\n    bias=\"none\",\n)\nmodel = get_peft_model(base_model, peft_config)\nmodel.config.use_cache = False\nmodel.print_trainable_parameters()\n\ncorpus = load_all_parallel_data()\nif not corpus:\n    raise SystemExit(\"[!] No training data found. Ensure internet is ON or datasets are attached.\")\n\n# Deterministic Split\nimport random\nrandom.seed(42)\nrandom.shuffle(corpus)\nsplit_idx = max(int(len(corpus) * 0.98), 1)\ntrain_rows = corpus[:split_idx]\neval_rows = corpus[split_idx:] if len(corpus) > 50 else corpus[:10]\nprint(f\"[+] Train dataset: {len(train_rows)} | Validation dataset: {len(eval_rows)}\")\n\ndef tokenize_batch(batch):\n    encoded = []\n    for source, target, src_l, tgt_l in zip(batch[\"source\"], batch[\"target\"], batch[\"src_lang\"], batch[\"tgt_lang\"]):\n        tokenizer.src_lang = src_l\n        tokenizer.tgt_lang = tgt_l\n        res = tokenizer(source, text_target=target, max_length=192, truncation=True)\n        encoded.append(res)\n    return {k: [r[k] for r in encoded] for k in encoded[0]}\n\ntrain_ds = Dataset.from_list(train_rows).map(tokenize_batch, batched=True, remove_columns=[\"source\", \"target\", \"src_lang\", \"tgt_lang\", \"tier\"])\neval_ds = Dataset.from_list(eval_rows).map(tokenize_batch, batched=True, remove_columns=[\"source\", \"target\", \"src_lang\", \"tgt_lang\", \"tier\"])\n\n# ---------------------------------------------------------------------------\n# Metrics: sacreBLEU and chrF++\n# ---------------------------------------------------------------------------\nbleu_metric = evaluate.load(\"sacrebleu\")\nchrf_metric = evaluate.load(\"chrf\")\n\ndef compute_metrics(eval_preds):\n    preds, labels = eval_preds\n    if isinstance(preds, tuple):\n        preds = preds[0]\n    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)\n    labels = [[l for l in label if l != -100] for label in labels]\n    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)\n\n    decoded_preds = [p.strip() for p in decoded_preds]\n    decoded_labels = [[l.strip()] for l in decoded_labels]\n\n    bleu = bleu_metric.compute(predictions=decoded_preds, references=decoded_labels)[\"score\"]\n    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)[\"score\"]\n    return {\"bleu\": round(bleu, 2), \"chrf\": round(chrf, 2)}\n\n# ---------------------------------------------------------------------------\n# Trainer Setup & Execution\n# ---------------------------------------------------------------------------\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\ncollator = DataCollatorForSeq2Seq(tokenizer, model=model)\n\ntraining_args = Seq2SeqTrainingArguments(\n    output_dir=str(OUTPUT_DIR / \"checkpoints\"),\n    num_train_epochs=int(os.getenv(\"NUM_EPOCHS\", \"3\")),\n    per_device_train_batch_size=int(os.getenv(\"BATCH_SIZE\", \"2\")),\n    per_device_eval_batch_size=2,\n    gradient_accumulation_steps=int(os.getenv(\"GRAD_ACCUM\", \"8\")),\n    learning_rate=float(os.getenv(\"LEARNING_RATE\", \"1e-4\")),\n    lr_scheduler_type=\"cosine\",\n    warmup_ratio=0.05,\n    weight_decay=0.01,\n    gradient_checkpointing=True,\n    fp16=torch.cuda.is_available(),\n    eval_strategy=\"epoch\",\n    save_strategy=\"epoch\",\n    save_total_limit=2,\n    logging_steps=50,\n    report_to=[\"none\"],\n    predict_with_generate=False,\n    load_best_model_at_end=True,\n    metric_for_best_model=\"eval_loss\",\n    greater_is_better=False,\n)\n\ntrainer = Seq2SeqTrainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_ds,\n    eval_dataset=eval_ds,\n    data_collator=collator,\n    compute_metrics=compute_metrics,\n)\n\nprint(\"=\" * 70)\nprint(f\"[*] Starting Master Kashmiri NLLB Training on {len(train_ds)} Parallel Pairs...\")\nprint(\"=\" * 70)\n\ntrainer.train()\n\nprint(\"[+] Training completed successfully!\")\neval_results = trainer.evaluate()\nprint(\"[+] Final Evaluation:\", eval_results)\n\n# ---------------------------------------------------------------------------\n# Generation Smoke Tests\n# ---------------------------------------------------------------------------\ndef translate_english_to_kashmiri(text: str) -> str:\n    tokenizer.src_lang = \"eng_Latn\"\n    inputs = tokenizer(text, return_tensors=\"pt\", truncation=True, max_length=192).to(model.device)\n    with torch.no_grad():\n        ids = model.generate(\n            **inputs,\n            forced_bos_token_id=tokenizer.convert_tokens_to_ids(\"kas_Arab\"),\n            max_new_tokens=128,\n            num_beams=4,\n            repetition_penalty=1.15,\n            no_repeat_ngram_size=3\n        )\n    return tokenizer.batch_decode(ids, skip_special_tokens=True)[0]\n\ntest_sentences = [\n    \"I am reading a book in Kashmiri.\",\n    \"The children are playing happily in the garden.\",\n    \"Peace and blessings be upon the Prophet.\",\n    \"Quantization is the process of mapping continuous values to discrete levels.\",\n    \"Kashmir is famous for its mountains, saffron, and beautiful lakes.\",\n]\n\nprint(\"\\n\" + \"=\" * 70)\nprint(\"MASTER TRANSLATION GENERATION SMOKE TEST:\")\nprint(\"=\" * 70)\nsmoke_results = []\nfor sent in test_sentences:\n    trans = translate_english_to_kashmiri(sent)\n    print(f\"EN: {sent}\")\n    print(f\"KS: {trans}\\n\")\n    smoke_results.append({\"english\": sent, \"kashmiri\": trans})\n\n# ---------------------------------------------------------------------------\n# Save & Hugging Face Upload\n# ---------------------------------------------------------------------------\nadapter_dir = OUTPUT_DIR / \"adapter\"\ntokenizer_dir = OUTPUT_DIR / \"tokenizer\"\nmodel.save_pretrained(str(adapter_dir))\ntokenizer.save_pretrained(str(tokenizer_dir))\n\nreport = {\n    \"base_model\": MODEL_ID,\n    \"total_parallel_pairs\": len(corpus),\n    \"evaluation_metrics\": eval_results,\n    \"smoke_test_generations\": smoke_results\n}\n(OUTPUT_DIR / \"training-report.json\").write_text(json.dumps(report, indent=2, ensure_ascii=False))\n\n# Create zip archive for 1-click download\nshutil.make_archive(\"/kaggle/working/kashmiri-nllb-master-adapter\", \"zip\", adapter_dir)\nprint(\"[+] Created download zip: /kaggle/working/kashmiri-nllb-master-adapter.zip\")\n\nif HF_TOKEN:\n    try:\n        from huggingface_hub import HfApi\n        api = HfApi(token=HF_TOKEN)\n        api.create_repo(repo_id=HF_REPO_ID, repo_type=\"model\", exist_ok=True)\n        api.upload_folder(\n            folder_path=str(adapter_dir),\n            repo_id=HF_REPO_ID,\n            repo_type=\"model\",\n            commit_message=\"Update: Master SOTA Kashmiri NLLB Translation LoRA Adapter (300k+ pairs)\"\n        )\n        print(f\"[+] Uploaded master adapter to: https://huggingface.co/{HF_REPO_ID}\")\n    except Exception as e:\n        print(f\"[-] Hugging Face upload skipped or error: {e}\")\n\nprint(\"=\" * 70)\nprint(\"MASTER KASHMIRI TRANSLATION PIPELINE COMPLETE!\")\nprint(\"=\" * 70)\n")
print("Self-contained script ready at:", script_dir / "finetune-nllb-master.py")


In [ ]:
# 3. Launch Master NLLB Kashmiri Translation Training
import os, subprocess

env = os.environ.copy()
env["BASE_MODEL"] = "facebook/nllb-200-distilled-600M"
env["NUM_EPOCHS"] = "3"
env["BATCH_SIZE"] = "2"
env["GRAD_ACCUM"] = "8"
env["LEARNING_RATE"] = "1e-4"
env["OUTPUT_DIR"] = "/kaggle/working/kashmiri-nllb-master"
env["HF_REPO_ID"] = "Sameer0313/kashmiri-nllb-lora"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# env["HF_TOKEN"] = "your_hf_token_here"

subprocess.run(["python", "scripts/finetune-nllb-master.py"], cwd=str(repo), env=env, check=True)


In [ ]:
# 4. Check Output & Generated Translation Report
from pathlib import Path
out = Path('/kaggle/working/kashmiri-nllb-master')
report_file = out / 'training-report.json'
if report_file.exists():
    print('\n================ MASTER TRAINING REPORT ================\n')
    print(report_file.read_text())
    print('========================================================')
zip_file = Path('/kaggle/working/kashmiri-nllb-master-adapter.zip')
if zip_file.exists():
    print(f'Download master adapter: {zip_file.name} ({zip_file.stat().st_size / 1024 / 1024:.2f} MB)')
